# Vector Stores

This file teaches you how to store embeddings in a database so you can search them later.
By the end, you will know how to use FAISS and ChromaDB, and when to use each one.

## Setup

In [1]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [4]:
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=api_key)

In [5]:
EMBED_MODEL = "text-embedding-3-small"

In [6]:
def get_embeddings(texts):
    """Get embeddings for a list of texts"""
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )

    embeddings = []
    for item in response.data:
        embeddings.append(item.embedding)
    return embeddings

**What happened:** We set up the OpenAI client and a helper function to create embeddings.

### Sample Documents

We will store these 8 documents in our vector databases.

In [7]:
DOCUMENTS = [
    {"text": "Kubernetes Pods are the smallest deployable units.",
     "source": "k8s-docs", "category": "core"},
    {"text": "HPA scales replicas based on CPU utilization.",
     "source": "k8s-docs", "category": "scaling"},
    {"text": "Services provide stable network endpoints.",
     "source": "k8s-docs", "category": "networking"},
    {"text": "RBAC restricts API access based on user roles.",
     "source": "k8s-docs", "category": "security"},
]

In [8]:
DOCUMENTS += [
    {"text": "Helm is a package manager for Kubernetes.",
     "source": "helm-docs", "category": "tooling"},
    {"text": "Istio service mesh manages microservice traffic.",
     "source": "istio-docs", "category": "networking"},
    {"text": "PersistentVolumes provide durable storage.",
     "source": "k8s-docs", "category": "storage"},
    {"text": "Container images are stored in registries.",
     "source": "docker-docs", "category": "core"},
]

In [9]:
texts = []
for doc in DOCUMENTS:
    texts.append(doc["text"])

print(len(texts))

embeddings = get_embeddings(texts)

8


**What happened:** We defined 8 sample documents about Kubernetes and created embeddings for all of them in one API call.

## FAISS (Facebook AI Similarity Search)

**What it does:** Stores embeddings in memory and searches them very fast.

**When to use it:** For prototyping or batch processing. FAISS is free, fast, and runs locally.

**Steps:**
1. Create a FAISS index with the right number of dimensions.
2. Add your embeddings to the index.
3. Search by passing a query embedding.

**FAISS index** is a data structure that holds your embeddings and can find the most similar ones quickly.

Install: `pip install faiss-cpu`

In [10]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 47.8 MB/s  0:00:00

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [36]:
import faiss

# Convert embeddings to a numpy matrix
emb_matrix = np.array(embeddings, dtype="float32")
print(emb_matrix)
print(len(emb_matrix))

shape_of_emb_matrix = emb_matrix.shape
print(shape_of_emb_matrix)

dimension = emb_matrix.shape[1]
print(dimension)

[[-0.02992249  0.03591919 -0.00134468 ... -0.00749207  0.01416779
   0.00099564]
 [-0.03344727  0.04528809  0.06033325 ... -0.05029297 -0.00494385
  -0.03619385]
 [-0.03259277 -0.02655029  0.06234741 ... -0.00314522 -0.04629517
   0.01384735]
 ...
 [ 0.00680161  0.01060486  0.0491333  ... -0.02073669 -0.05874634
   0.00265312]
 [ 0.00440216  0.03198242  0.04318237 ... -0.01402283  0.01521301
   0.00091171]
 [ 0.0069809   0.02078247  0.05831909 ... -0.00906372  0.01843262
   0.00144863]]
8
(8, 1536)
1536


In [24]:
# Create and fill the index

index = faiss.IndexFlatL2(dimension)
index.add(emb_matrix)

print(f"Vectors stored: {index.ntotal}")

Vectors stored: 8


**What happened:** We created a FAISS index and added 8 embeddings to it. `IndexFlatL2` compares every query against every stored vector (brute force).

In [29]:
query = "How does Kubernetes autoscale?"

query_embedding = get_embeddings([query])
print(query_embedding)

query_array = np.array(query_embedding, dtype="float32")
print(query_array)

[[-0.032135009765625, 0.020782470703125, 0.0657958984375, 0.019256591796875, -3.36766242980957e-05, -0.01020050048828125, -0.03558349609375, -0.01715087890625, -0.03955078125, 0.0299224853515625, -0.0206298828125, 0.007633209228515625, -0.0269927978515625, 0.00533294677734375, -0.00843048095703125, -0.025238037109375, -0.005680084228515625, 0.0008549690246582031, -0.056427001953125, -0.02545166015625, -0.0007205009460449219, -0.0183868408203125, 0.0274200439453125, -0.01227569580078125, -0.0304412841796875, 0.0143280029296875, -0.0029087066650390625, 0.0006585121154785156, 0.0091705322265625, -0.026214599609375, -0.00011652708053588867, -0.03533935546875, -0.045074462890625, -0.014678955078125, 0.004352569580078125, -0.0225372314453125, -0.0006961822509765625, 0.040191650390625, -0.01349639892578125, 0.025665283203125, 0.00189971923828125, -0.0399169921875, -0.03131103515625, 0.055419921875, -0.011260986328125, -0.018218994140625, 0.0372314453125, -0.0166168212890625, 0.031829833984375

In [31]:
distances, indices = index.search(query_array, k=3)
print(distances)
print(indices)

[[1.0288397 1.2052606 1.2564334]]
[[0 4 5]]


In [33]:
for i in range(len(distances[0])):
    dist = distances[0][i]
    idx = indices[0][i]

    doc_text = DOCUMENTS[idx]["text"]

    print(f"{dist:.3f} - {doc_text}")

1.029 - Kubernetes Pods are the smallest deployable units.
1.205 - Helm is a package manager for Kubernetes.
1.256 - Istio service mesh manages microservice traffic.


**What happened:** We searched for "How does Kubernetes autoscale?" and got the 3 most similar documents. Lower distance means more similar. The HPA document ranked first because it is about autoscaling.

**FAISS limitation:** FAISS only stores numbers. It does not store the original text or metadata (like source or category). You must keep that information in a separate list and use the index number to look it up.

## ChromaDB

**What it does:** Stores embeddings, text, and metadata together. Supports filtering by metadata.

**When to use it:** For development and small production systems. It is easier to use than FAISS because it stores everything in one place.

**Steps:**
1. Create a ChromaDB client and a collection.
2. Add documents with their embeddings and metadata.
3. Search by passing a query embedding. You can also filter by metadata.

Install: `pip install chromadb`

In [38]:
!pip install chromadb


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [39]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name = "k8s_docs",
    metadata={
        "hnsw:space": "cosine"
    }
)

**What happened:** We created an in-memory ChromaDB client and a collection called "k8s_docs" that uses cosine similarity.

In [40]:
doc_ids = []

for i in range(len(DOCUMENTS)):
    doc_ids.append(f"doc_{i}")


metadata_list = []
for doc in DOCUMENTS:
    meta = {
        "source": doc["source"],
        "category": doc["category"]
    }

    metadata_list.append(meta)

In [42]:
collection.add(
    ids=doc_ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadata_list
)

print(collection.count())

8


**What happened:** We added all 8 documents to ChromaDB. Each document has its text, embedding, and metadata (source and category) stored together.

In [53]:
# Basic search

query = "How does Kubernetes autoscale?"

query_embedding = get_embeddings([query])

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(results)

{'ids': [['doc_0', 'doc_4', 'doc_5']], 'embeddings': None, 'documents': [['Kubernetes Pods are the smallest deployable units.', 'Helm is a package manager for Kubernetes.', 'Istio service mesh manages microservice traffic.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'core', 'source': 'k8s-docs'}, {'source': 'helm-docs', 'category': 'tooling'}, {'category': 'networking', 'source': 'istio-docs'}]], 'distances': [[0.5144471526145935, 0.6029576063156128, 0.628382682800293]]}


In [54]:
num_results = len(results["documents"][0])
print(num_results)

3


In [55]:
for i in range(num_results):
    doc_text = results["documents"][0][i]
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]

    print(f"{doc_text} - {meta} - {dist}")

Kubernetes Pods are the smallest deployable units. - {'category': 'core', 'source': 'k8s-docs'} - 0.5144471526145935
Helm is a package manager for Kubernetes. - {'source': 'helm-docs', 'category': 'tooling'} - 0.6029576063156128
Istio service mesh manages microservice traffic. - {'category': 'networking', 'source': 'istio-docs'} - 0.628382682800293


## Metadata Filtering

**What it does:** Narrows your search to only documents that match certain metadata values.

**When to use it:** When you want to search within a specific category, source, or date range.

**Steps:**
1. Add a `where` parameter to your query.
2. ChromaDB only searches documents that match the filter.
3. You get results that are both relevant and in the right category.

In [56]:
# Search only networking documents 

search_text = "network communication between services"
search_embedding = get_embeddings([search_text])

results = collection.query(
    query_embeddings=search_embedding,
    n_results=3,
    where={
        "category": "networking"
    }
)

print(results)

{'ids': [['doc_2', 'doc_5']], 'embeddings': None, 'documents': [['Services provide stable network endpoints.', 'Istio service mesh manages microservice traffic.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'networking', 'source': 'k8s-docs'}, {'source': 'istio-docs', 'category': 'networking'}]], 'distances': [[0.5290471315383911, 0.5834792852401733]]}


In [57]:
num_results = len(results["documents"][0])


for i in range(num_results):
    doc_text = results["documents"][0][i]
    meta = results["metadatas"][0][i]
    
    print(f"{doc_text} - {meta}")

Services provide stable network endpoints. - {'category': 'networking', 'source': 'k8s-docs'}
Istio service mesh manages microservice traffic. - {'source': 'istio-docs', 'category': 'networking'}


**What happened:** We filtered the search to only networking documents. ChromaDB skipped all documents with other categories.

In [58]:
# Complex filter: k8s-docs AND not security

search_text = "Kubernetes concepts"
search_embedding = get_embeddings([search_text])

results = collection.query(
    query_embeddings=search_embedding,
    n_results=3,
    where={
        "$and": [
            {
                "source": {
                    "$eq": "k8s-docs"
                }
            },
            {
                "category": {
                    "$ne": "security"
                }
            }
        ]
    }
)

print(results)

{'ids': [['doc_0', 'doc_2', 'doc_6']], 'embeddings': None, 'documents': [['Kubernetes Pods are the smallest deployable units.', 'Services provide stable network endpoints.', 'PersistentVolumes provide durable storage.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'k8s-docs', 'category': 'core'}, {'source': 'k8s-docs', 'category': 'networking'}, {'source': 'k8s-docs', 'category': 'storage'}]], 'distances': [[0.4544089436531067, 0.6828231811523438, 0.7207309603691101]]}


In [59]:
num_results = len(results["documents"][0])

for i in range(num_results):
    meta = results["metadatas"][0][i]
    doc_text = results["documents"][0][i]
    
    print(f"{doc_text} - {meta}")

Kubernetes Pods are the smallest deployable units. - {'source': 'k8s-docs', 'category': 'core'}
Services provide stable network endpoints. - {'source': 'k8s-docs', 'category': 'networking'}
PersistentVolumes provide durable storage. - {'source': 'k8s-docs', 'category': 'storage'}


**What happened:** We used `$and`, `$eq`, and `$ne` operators to combine two filters. Only documents from k8s-docs that are NOT in the security category were searched.

In [60]:
# cleanup

chroma_client.delete_collection("k8s_docs")

## Index Types

Vector databases use different data structures to store and search embeddings. Here are the main types:

| Type | How it works | Speed | Accuracy | Best for |
|------|-------------|-------|----------|----------|
| **Flat** | Compares query against every vector | Slow for large data | 100% | Less than 100K vectors |
| **IVF** | Groups vectors into clusters, searches nearby clusters only | Medium | 95-99% | 100K to 10M vectors |
| **HNSW** | Builds a graph connecting similar vectors | Fast | 95-99% | Any size, best general choice |
| **PQ** | Compresses vectors into smaller codes | Fast + low memory | 90-95% | Billions of vectors |

## Vector Store Comparison

| Store | Type | Hosting | Best For |
|-------|------|---------|----------|
| **FAISS** | Library | Local | Prototyping, batch search |
| **ChromaDB** | Database | Local | Development, small production |
| **Pinecone** | Database | Cloud | Managed, scalable production |
| **Weaviate** | Database | Both | Built-in hybrid search |
| **pgvector** | Extension | Any PostgreSQL | Teams already using PostgreSQL |

**How to choose:**
- Prototype / learning: ChromaDB (simplest API)
- Already using PostgreSQL: pgvector (no new infrastructure)
- Need managed + scale: Pinecone (easiest cloud option)
- Need hybrid search: Weaviate (combines dense and sparse natively)

## Summary

- **FAISS** is a fast local library for vector search. It stores only numbers, not text or metadata.
- **ChromaDB** stores embeddings, text, and metadata together. It supports metadata filtering.
- **Metadata filtering** lets you narrow search results by category, source, or other fields.
- **Index types** (Flat, IVF, HNSW, PQ) trade off speed, accuracy, and memory usage.
- Use ChromaDB for prototyping and Pinecone or pgvector for production.